In [ ]:
try:
    import claveles  # noqa: F401
except ImportError:
    %pip install claveles

**Note for Google Colab Users:**
During installation, Colab may display a dependency conflict error regarding `protobuf` (`ERROR: pip's dependency resolver...`). This occurs due to pre-installed Google packages in the Colab environment. 

As long as you see `Successfully installed ...` at the end of the log, the required packages are properly installed. **Please ignore the error and continue running the rest of the notebook.**

$$
\newcommand{\bra}[1]{{\left\langle{#1}\right|}}
\newcommand{\ket}[1]{{\left|{#1}\right\rangle}}
\newcommand{\Bra}[1]{{\Big\langle{#1}\Big|}}
\newcommand{\Ket}[1]{{\Big|{#1}\Big\rangle}}
\newcommand{\braket}[1]{{\left\langle{#1}\right\rangle}}
\newcommand{\dag}{\dagger}
$$

# Hong–Ou–Mandel Interference

## Introduction

The Hong–Ou–Mandel (HOM) effect is a foundational phenomenon in quantum optics: when two single photons enter a 50:50 beamsplitter from opposing input ports and interact on it, they invariably exit together through the *same* output port. Counter-intuitively, the probability that the two photons get simultaneously detected in both of the outputs drops to zero, as such probability amplitude is cancelled out due to quantum interference. This tutorial demonstrates how the *indistinguishability* of the two photons drives this non-classical interference, leading to the phenomenon known as *photon bunching*.

Since this effect involves counting individual photons, it is most naturally described in the *Fock basis*, making it an ideal introductory example for Fock-space numerical simulations—essentially a "Hello World" for working with photon-number states.

---

## Background

### Indistinguishability

The HOM effect is a quantum interference phenomenon that manifests when two photons, each prepared in the single-photon Fock state $\ket{1}$, are completely indistinguishable. In other words, these photons share identical physical properties (such as frequency, polarization, spatial mode, and temporal wave packet) and exist in the exact same quantum profile, making it fundamentally impossible to tell which photon is which once they mix. Their interference at a beamsplitter cannot be explained by classical electrodynamics; it is a direct consequence of the bosonic statistics of light.

### Initial State

We simultaneously launch exactly one photon into each of the two input ports (modes 0 and 1). The total input state is given by the product of two single-photon Fock states:

$$
\ket{\psi_\text{in}} = \ket{1}_0 \ket{1}_1,
$$

where the subscript labels the specific optical mode, such that $\ket{n}_i$ denotes that mode $i$ contains $n$ photons. We index the modes starting from 0 to align with the array indices used in the simulation code below (`Q(0, 1)`).

### Beamsplitter Operator

A beamsplitter acting on modes 0 and 1 is described by the unitary operator:

$$
\hat{B}(\theta, \phi) = \exp\!\left[\theta\left(e^{i\phi}\, \hat{a}_0 \hat{a}_1^\dagger - e^{-i\phi}\, \hat{a}_0^\dagger \hat{a}_1\right)\right],
$$

where $\hat{a}_i$ and $\hat{a}_i^\dagger$ are the annihilation and creation operators for mode $i$, respectively. A symmetric 50:50 beamsplitter corresponds to $\theta = \pi/4$ and $\phi = 0$, ensuring a symmetric mixing of the two modes. To maintain readability in the subsequent derivations, we write this operator simply as $\hat{B}$ when its parameters are clear from context.

For any choice of parameters $\theta$ and $\phi$, the beamsplitter merely redistributes photons between the two active modes without creating or destroying them, thereby conserving the total photon number. Mathematically, $\hat{B}$ commutes with the *total photon number operator* $\hat{N} = \hat{a}_0^\dagger \hat{a}_0 + \hat{a}_1^\dagger \hat{a}_1$, where $\hat{a}_i^\dagger \hat{a}_i$ is the *number operator* for mode $i$, satisfying $\hat{a}_i^\dagger \hat{a}_i \ket{n}_i = n \ket{n}_i$. Concretely, because $[\hat{B}, \hat{N}] = 0$, we have:

$$
\hat{B}^\dagger\,\hat{N}\,\hat{B} = \hat{B}^\dagger\,\hat{B}\,\hat{N} = \hat{N},
$$

implying that the total photon number is strictly conserved (i.e., for any quantum state $\ket{\phi} = \hat{B}\ket{\psi}$, $\braket{\phi|\hat{N}|\phi}=\braket{\psi|\hat{N}|\psi}$ holds). Consequently, every constituent component of the output quantum state contains exactly two photons.

### Output State and Zero Coincidence

Propagating the input state $\ket{1}_0\ket{1}_1$ through a 50:50 beamsplitter $\hat{B}(\pi/4, 0)$ yields:

$$
\hat{B}(\pi/4, 0)\,\ket{1}_0 \ket{1}_1 = \frac{1}{\sqrt{2}}\left(\ket{0}_0 \ket{2}_1 - \ket{2}_0 \ket{0}_1\right),
$$

as derived in the [Appendix](#appendix-derivation-of-the-output-state). The defining feature of this result is the complete absence of the $\ket{1}_0\ket{1}_1$ term. Denoting $n_i$ as the number of photons detected in mode $i$, the probability of registering one photon at each output port (a *coincidence event*) is identically zero:

$$
\mathrm{Prob}(n_0 = 1, n_1 = 1) = \left|\bra{1_0 1_1}\, \hat{B}(\pi/4, 0)\, \ket{1_0 1_1}\right|^2 = 0,
$$

ensuring that both photons always emerge together from the same output port, but never one photon per port. 

| Port 0 | Port 1 | Probability |
| :---: | :---: | :---: |
| 0 | 2 | $\frac{1}{2}$ |
| 1 | 1 | 0 |
| 2 | 0 | $\frac{1}{2}$ |

Quantum states of this form, $\frac{1}{\sqrt{2}}(\ket{0,2} - \ket{2,0})$, are referred to as *NOON states* (configured here for $N = 2$).

If the two photons are distinguishable, then the physics here suddenly collapses to classical probabilities. A balanced beamsplitter has a 50/50 chance of transmitting or reflecting a photon; thus, two non-interacting photons are individually equally likely to exit from either port, resulting in a probability table as follows:


| Port 0 | Port 1 | Probability |
| :---: | :---: | :---: |
| 0 | 2 | $\frac{1}{4}$ |
| 1 | 1 | $\frac{1}{2}$  |
| 2 | 0 | $\frac{1}{4}$ |

On the other hand, if the two photons are indistinguishable, then the probability amplitude of $\ket{1}_0\otimes\ket{1}_1$ completely cancels out due to quantum coherence. This phenomenon is known as *photon bunching* and strongly reflects the quantum nature of photons. 

We verify this behavior by evaluating the *reduced density matrix* of each individual mode, obtained by taking the partial trace over the companion mode. Two core metrics are sufficient: the per-mode photon-number distributions—defined by the diagonal elements of the reduced density matrices—and the principle of photon-number conservation. As shown below, these metrics uniquely determine the coincidence and bunching probabilities, providing a direct quantitative measurement of the effect.

---

## 1. Setup

First, let us import the required libraries and simulation components.

In [ ]:
import numpy as np
from claveles.circuit import CircuitRepr
from claveles.circuit.ops.std import BeamSplitter
from claveles.circuit.state import QuantumState
from claveles.client import SimulatorClient
from claveles.execute import execute

We perform our calculations within a truncated Fock space of dimension `PHOTON_CUTOFF` (spanning photon numbers $0, 1, \dots, D-1$). Because photon-number conservation strictly caps the maximum photon count in any single mode at the total input value of two, the states $\ket{0}, \ket{1}, \ket{2}$ fully span the accessible Hilbert space. Consequently, a cutoff of $D = 3$ models the underlying quantum dynamics exactly, introducing zero truncation error.

In [ ]:
PHOTON_CUTOFF = 3

---

## 2. Defining the Initial State

Each input mode is initialized in the single-photon Fock state $\ket{1}$ using the `QuantumState.fock(n, cutoff_dim=...)` factory. The full initial state is thus given by $\ket{1} \otimes \ket{1}$.

In [ ]:
input_state_0 = QuantumState.fock(1, cutoff_dim=PHOTON_CUTOFF)
input_state_1 = QuantumState.fock(1, cutoff_dim=PHOTON_CUTOFF)

---

## 3. Applying the Beamsplitter

We apply a single 50:50 beamsplitter across the two modes. The Claveles gate `BeamSplitter(theta, phi)` implements the operator $\hat{B}(\theta, \phi)$ introduced in the background section: it transforms the creation operator of mode 0 as $\hat{B}\,\hat{a}_0^\dagger\,\hat{B}^\dagger = \cos\theta\,\hat{a}_0^\dagger + e^{i\phi}\sin\theta\,\hat{a}_1^\dagger$, so a photon entering mode 0 stays in mode 0 with probability $\cos^2\theta$ and moves to mode 1 with probability $\sin^2\theta$. A 50:50 beamsplitter needs $\cos^2\theta = \sin^2\theta = 1/2$, so we set $\theta = \pi/4$, and $\phi = 0$ yields the symmetric beamsplitter $\hat{B}(\pi/4, 0)$.

The circuit first places the single-photon states on modes 0 and 1 (a state has to be the first thing that touches its mode) and then applies the beamsplitter.

In [ ]:
circuit = CircuitRepr("hom")
circuit.Q(0) | input_state_0
circuit.Q(1) | input_state_1
circuit.Q(0, 1) | BeamSplitter(theta=np.pi / 4, phi=0)

---

## 4. Running the Simulation

We execute the quantum circuit using the local Fock-basis numerical simulator. Because we only require the final quantum state vector (rather than sampled measurement statistics), a single simulation shot is sufficient. The output state is available as `result.client_result.shot_results[0].state`, a `QuantumState` holding the joint state of both modes. Its `extract_mode(m)` method traces out the other mode and returns mode $m$ alone, and the `dm()` method of that single-mode state returns the reduced density matrix of mode $m$ as a $D \times D$ matrix, where the diagonal element $[n, n]$ corresponds directly to the probability $\mathrm{Prob}(N = n)$ of finding exactly $n$ photons in that mode.

In [ ]:
client = SimulatorClient(n_shots=1, backend="cpu", state_save_policy="first_only")
result = execute(circuit, client)
output_state = result.client_result.shot_results[0].state

rho_0 = output_state.extract_mode(0).dm()
rho_1 = output_state.extract_mode(1).dm()

---

## 5. Verifying the Result

The diagonal entries of each reduced density matrix represent the underlying photon-number distribution for that specific mode. Theoretically, we expect each mode to contain either zero or two photons with equal probability, while the probability of containing exactly one photon should vanish.

In [ ]:
prob_0 = np.real(np.diag(rho_0))
prob_1 = np.real(np.diag(rho_1))

print("Photon-number distribution P(N) per mode:")
print(f"  mode 0: {np.round(prob_0, 3)}")
print(f"  mode 1: {np.round(prob_1, 3)}")

The simulation results show that each mode satisfies $\mathrm{Prob}(N = 0) = \mathrm{Prob}(N = 2) = 0.5$ and $\mathrm{Prob}(N = 1) = 0$. From these marginal statistics, we can directly deduce the two defining hallmarks of the HOM effect: vanishing coincidence and complete photon bunching. First, the coincidence rate vanishes because registering a single photon in mode 0 requires $N_0 = 1$, an event with zero probability:

In [ ]:
p_single_photon_0 = prob_0[1]
p_single_photon_1 = prob_1[1]

print(f"P(N_0 = 1) = {p_single_photon_0:.3f}  (expected 0)")
print(f"P(N_1 = 1) = {p_single_photon_1:.3f}  (expected 0)")

Second, the principle of photon-number conservation uniquely determines the remaining joint probabilities. The beamsplitter strictly conserves the total photon number, which is constrained to $N_0 + N_1 = 2$ by the initial state $\ket{1,1}$; thus, every non-vanishing component of the output state must contain exactly two photons. The only two-photon configurations compatible with our marginal observation that neither mode can contain a single photon are $\ket{2,0}$ and $\ket{0,2}$. The coincidence term $\ket{1,1}$ is strictly suppressed because $\mathrm{Prob}(N = 1) = 0$ for both subsystems. Consequently, the states $\ket{2,0}$ and $\ket{0,2}$ each inherit exactly half of the total probability mass:

In [ ]:
p_coincidence = p_single_photon_0  # Prob(N_0 = 1, N_1 = 1)
p_bunching = prob_0[0] + prob_0[2]  # Prob(2,0) + Prob(0,2) via conservation

print(f"Coincidence P(1,1)        = {p_coincidence:.3f}  (expected 0)")
print(f"Bunching P(2,0) + P(0,2)  = {p_bunching:.3f}  (expected 1)")

The coincidence probability $\mathrm{Prob}(N_0 = 1, N_1 = 1)$ evaluates to zero, while the entire probability distribution concentrates on the bunched configurations $\ket{2,0}$ and $\ket{0,2}$. This perfectly reproduces the classic HOM effect: an absolute suppression of coincidences paired with complete photon bunching, demonstrating precise quantitative agreement with the theoretically predicted state $\frac{1}{\sqrt{2}}(\ket{0,2} - \ket{2,0})$.

---

## Summary

In this tutorial, we have demonstrated the following:

1. **Initial state configuration**: Two completely indistinguishable photons are prepared in the product Fock state $\ket{1} \otimes \ket{1}$ using the `QuantumState.fock` factory.
2. **Beamsplitter quantum interference**: Applying a symmetric 50:50 beamsplitter gate ($\theta = \pi/4$, $\phi = 0$) to the target modes generates the entangled NOON state $\frac{1}{\sqrt{2}}(\ket{0,2} - \ket{2,0})$.
3. **Characterization of photon bunching**: Analyzing the reduced density matrix for each individual subsystem reveals that $\mathrm{Prob}(N = 1) = 0$, ensuring the complete suppression of the coincidence probability $\mathrm{Prob}(N_0 = 1, N_1 = 1) = 0$. Combined with the strict requirement of photon-number conservation ($N_0 + N_1 = 2$), this proves that the two input photons invariably bunch into the same output port. This numerically verifies the Hong–Ou–Mandel effect in strict quantitative agreement with the analytical prediction.

The discrete photon-counting framework of the Fock representation makes the non-classical interference of indistinguishable photons directly observable within the per-mode statistics.

---

## Appendix: Derivation of the Output State

In this section, we derive the output quantum state $\hat{B}(\pi/4, 0)\,\ket{1}_0\ket{1}_1 = \frac{1}{\sqrt{2}}(\ket{0,2} - \ket{2,0})$ introduced in the background section. An elegant way to trace this transformation is to utilize the unitarity of the beamsplitter operator $\hat{B} \equiv \hat{B}(\pi/4, 0)$ to rewrite the input creation operators. Conjugating the individual creation operators by $\hat{B}$ yields:

$$
\hat{B}\,\hat{a}_0^\dagger\,\hat{B}^\dagger = \frac{\hat{a}_0^\dagger + \hat{a}_1^\dagger}{\sqrt{2}}, \qquad
\hat{B}\,\hat{a}_1^\dagger\,\hat{B}^\dagger = \frac{\hat{a}_1^\dagger - \hat{a}_0^\dagger}{\sqrt{2}}.
$$

Expressing the input state as $\ket{1}_0\ket{1}_1 = \hat{a}_0^\dagger \hat{a}_1^\dagger \ket{0,0}$ and inserting the identity operator $\hat{B}^\dagger \hat{B} = \hat{I}$ between the creation operators (noting that the vacuum state is invariant, $\hat{B}\ket{0,0} = \ket{0,0}$), we find:

$$
\hat{B}\,\ket{1}_0\ket{1}_1
= \left(\hat{B}\,\hat{a}_0^\dagger\,\hat{B}^\dagger\right)\left(\hat{B}\,\hat{a}_1^\dagger\,\hat{B}^\dagger\right)\ket{0,0}
= \frac{1}{2}\left(\hat{a}_0^\dagger + \hat{a}_1^\dagger\right)\left(\hat{a}_1^\dagger - \hat{a}_0^\dagger\right)\ket{0,0}.
$$

The two cross-product terms involving $\hat{a}_0^\dagger \hat{a}_1^\dagger$ cancel exactly due to opposing signs, leaving the simplified expression $\frac{1}{2}\left((\hat{a}_1^\dagger)^2 - (\hat{a}_0^\dagger)^2\right)\ket{0,0}$. Applying the relation $(\hat{a}_i^\dagger)^2\ket{0} = \sqrt{2}\,\ket{2}$ yields the target state:

$$
\hat{B}(\pi/4, 0)\,\ket{1}_0 \ket{1}_1 = \frac{1}{\sqrt{2}}\left(\ket{0}_0 \ket{2}_1 - \ket{2}_0 \ket{0}_1\right).
$$

The two probability amplitudes that would physically correspond to registering a single photon at each output port are precisely the cross terms that cancel out in the operator algebra, ensuring that the coincidence component $\ket{1}_0\ket{1}_1$ vanishes completely.

---

## References

- \[1\] C. K. Hong, Z. Y. Ou, L. Mandel, *Measurement of subpicosecond time intervals between two photons by interference*, [Phys. Rev. Lett. **59**, 2044 (1987)](https://doi.org/10.1103/PhysRevLett.59.2044).